# Phase 2 — Fine-tune Qwen3-8B (LoRA bf16) trên VLSP DateArith VI, sau đó đánh giá lại

Notebook này thực hiện 2 bước:
1. **TRAIN**: SFT bằng LoRA (r=16, bf16) trên rows 1500–2999 của `date_training_dataset.txt` (1500 rows), chia 90/10. Adapter lưu local `/content/checkpoints/lora_vlsp_date_bf16/` (mất khi Colab disconnect — đúng lựa chọn của user).
2. **EVAL**: chạy lại 3 phương pháp (`zero_shot`, `few_shot k=3`, `dynamic_few_shot k=3`) trên 2 dataset (`vlsp_date` in-domain, `vlsp_duration` cross-task) với adapter → 6 run.

Output FT được ghi vào `outputs_ft/` (tách hẳn với `outputs/` baseline) để dễ so sánh.

⚠️ **Data leakage warning** với `dynamic_few_shot`: pool retrieval = rows 1500–2999 — trùng với tập train. Kết quả dynamic sau FT sẽ bias upward, ghi nhớ khi report.

## Setup

In [ ]:
# === SETUP 1 — cài môi trường (thêm peft / trl / datasets cho training) ===
!pip install -q -U transformers accelerate scikit-learn pyyaml sentence-transformers faiss-cpu
!pip install -q -U peft trl datasets
# ⚠️ Sau khi chạy cell này: Runtime → Restart session rồi chạy tiếp từ SETUP 2.

In [ ]:
# === SETUP 2 — tùy chọn mount Drive + clone repo ===
import os

USE_DRIVE = False  # True: mount Drive để dùng Dataset/output trên Drive
REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: đổi
REPO_DIR = '/content/Temporal_Reasoning'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())
print('USE_DRIVE:', USE_DRIVE)

In [ ]:
# === SETUP 3 — paths: Dataset, output FT, checkpoint local ===
import os

if USE_DRIVE:
    DRIVE_OUT_FT = '/content/drive/MyDrive/Temporal_Reasoning/outputs_ft'
    DATASET_ROOT = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    DRIVE_OUT_FT = '/content/outputs_ft'
    DATASET_ROOT = os.path.join(REPO_DIR, 'Dataset')
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    if not os.path.exists(DATASET_ROOT):
        raise FileNotFoundError(f'Không tìm thấy Dataset tại {DATASET_ROOT}.')
    print('Dataset ->', DATASET_ROOT)

# Checkpoint local Colab session (mất khi disconnect — đúng lựa chọn user).
CHECKPOINT_DIR = '/content/checkpoints/lora_vlsp_date_bf16'
os.makedirs(os.path.dirname(CHECKPOINT_DIR), exist_ok=True)

print('DRIVE_OUT_FT :', DRIVE_OUT_FT)
print('CHECKPOINT   :', CHECKPOINT_DIR)

In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

## TRAIN — SFT LoRA bf16

Thời gian dự kiến trên A100 (40GB): ~10–15 phút (3 epoch × 1350/8 ≈ 506 steps).

Smoke quick (nếu muốn test pipeline): set `cfg.num_epochs=1, cfg.train_pool_size=50` trước khi train_sft.

In [ ]:
# === TRAIN — chạy SFT LoRA, save adapter vào CHECKPOINT_DIR ===
import yaml
from src.training.sft import SFTRunConfig, train_sft

with open('configs/sft_vlsp_date_lora_bf16.yaml', encoding='utf-8') as f:
    raw = yaml.safe_load(f)
raw['output_dir'] = CHECKPOINT_DIR  # override về path local Colab
cfg = SFTRunConfig(**raw)
print('SFT config:', cfg)

ADAPTER_PATH = train_sft(cfg)
print('Adapter saved to:', ADAPTER_PATH)
!ls -lah $ADAPTER_PATH

## EVAL — chạy lại 3 phương pháp × 2 dataset với adapter

Load model (base + adapter) **một lần** để reuse cho 6 run — tiết kiệm thời gian.
Output FT ghi vào `outputs_ft/`, summary.csv riêng → dễ diễn tả baseline vs FT.

In [ ]:
# === EVAL setup — load Qwen3-8B + adapter 1 lần ===
from src.models.qwen import QwenChatLM, QwenConfig
from src.runner import load_config, run

MODEL_FT = QwenChatLM(QwenConfig(
    model_name='Qwen/Qwen3-8B',
    dtype='bfloat16',
    adapter_path=ADAPTER_PATH,
))
MODEL_FT.load()
print('Model + adapter ready:', MODEL_FT.config.model_name, '+', MODEL_FT.config.adapter_path)

def run_exp_ft(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
               running_score_every=100, output_dir=DRIVE_OUT_FT):
    """Helper: load config, suffix experiment_name='_ft', trỏ về outputs_ft/, chạy với MODEL_FT."""
    cfg = load_config(cfg_path)
    cfg.output_dir = output_dir
    cfg.experiment_name = cfg.experiment_name + '_ft'
    cfg.adapter_path = ADAPTER_PATH  # log vào summary.csv
    cfg.verbose = verbose
    cfg.verbose_first_n = verbose_first_n
    cfg.verbose_every = verbose_every
    cfg.running_score_every = running_score_every
    return run(cfg, model=MODEL_FT)

### `vlsp_date` (in-domain) — kỳ vọng accuracy tăng rõ rệt so với baseline (zero=26.53%, few=36.53%, dynamic=55.6%).

In [ ]:
# === EXP 1/6 — zero_shot × vlsp_date (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 2/6 — few_shot k=3 × vlsp_date (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 3/6 — dynamic_few_shot k=3 × vlsp_date (FT)
# ⚠️ Data leakage: pool retrieval = rows 1500–2999 = train pool. Kết quả cần note. ===
m = run_exp_ft('configs/dynamic_few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### `vlsp_duration` (cross-task, cùng VI) — kiểm tra catastrophic forgetting.

In [ ]:
# === EXP 4/6 — zero_shot × vlsp_duration (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 5/6 — few_shot k=4 × vlsp_duration (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 6/6 — dynamic_few_shot k=4 × vlsp_duration (FT) ===
m = run_exp_ft('configs/dynamic_few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

## So sánh baseline vs FT

In [ ]:
# === COMPARE — join baseline summary với FT summary trên (method, dataset) ===
import os
import pandas as pd

BASE_SUMMARY = '/content/outputs/summary.csv' if not USE_DRIVE \
    else '/content/drive/MyDrive/Temporal_Reasoning/outputs/summary.csv'
FT_SUMMARY = os.path.join(DRIVE_OUT_FT, 'summary.csv')

def _load(path, label):
    if not os.path.exists(path):
        print(f'[warn] không tìm thấy {label} summary: {path}')
        return None
    df = pd.read_csv(path)
    keep = ['method', 'dataset', 'k_shot', 'metric', 'score', 'parse_fail']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.columns = [c if c in ('method', 'dataset', 'k_shot', 'metric') else f'{c}_{label}' for c in df.columns]
    return df

base = _load(BASE_SUMMARY, 'base')
ft = _load(FT_SUMMARY, 'ft')
if base is not None and ft is not None:
    cmp = base.merge(ft, on=['method', 'dataset', 'k_shot', 'metric'], how='outer')
    cmp['delta'] = cmp['score_ft'] - cmp['score_base']
    print(cmp.sort_values(['dataset', 'method']).to_string(index=False))
else:
    print('Skip compare — thiếu summary file.')

In [ ]:
# === EXPORT — zip outputs_ft để tải về máy trước khi Colab disconnect ===
import os
import shutil
from google.colab import files

os.chdir('/content')
out_dir = os.path.abspath(DRIVE_OUT_FT)
if not os.path.isdir(out_dir):
    raise FileNotFoundError(out_dir)
parent_dir = os.path.dirname(out_dir)
folder_name = os.path.basename(out_dir.rstrip('/'))
zip_path = shutil.make_archive(f'/content/{folder_name}', 'zip', root_dir=parent_dir, base_dir=folder_name)
print('Created zip:', zip_path)
files.download(zip_path)